## Web Scrape the recent financial news. . 

### Step-1-> Get the websites. . 

In [6]:
import requests

In [7]:
# base url of duckduckgo search
base_url="https://html.duckduckgo.com/html/"
# search query
search_query="financial news"

# headers to mimic a browser request
headers = {
  "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36"
}

# parameters for the search query
params = {
    "q": search_query,
    "s": "0",
    "dc": "",
    "vqd": "3---",
    "o": "json", 
    "api": "d.js",
    "kl": "us-en",
    "ct": "us-en",
    "ia": "web"
}
response=requests.get(base_url,params=params,headers=headers)
print(response.text)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<!--[if IE 6]><html class="ie6" xmlns="http://www.w3.org/1999/xhtml"><![endif]-->
<!--[if IE 7]><html class="lt-ie8 lt-ie9" xmlns="http://www.w3.org/1999/xhtml"><![endif]-->
<!--[if IE 8]><html class="lt-ie9" xmlns="http://www.w3.org/1999/xhtml"><![endif]-->
<!--[if gt IE 8]><!--><html xmlns="http://www.w3.org/1999/xhtml"><!--<![endif]-->
<head>
  <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=3.0, user-scalable=1" />
  <meta name="referrer" content="origin" />
  <meta name="HandheldFriendly" content="true" />
  <meta name="robots" content="noindex, nofollow" />
  <title>financial news at DuckDuckGo</title>
  <link title="DuckDuckGo (HTML)" type="application/opensearchdescription+xml" rel="search" href="//duckduckgo.com/opensearch_html_v2.xml" />
  <link hre

In [8]:
# using BeautifulSoup to parse the HTML response
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')


# extracting search results

result_elements = soup.select("#links .result")
serp_results = []
for result in result_elements:
    title_element = result.select_one(".result__a")
    url = "https:" + title_element["href"]
    title = title_element.get_text(strip=True)

    url_element = result.select_one(".result__url")
    display_url = url_element.get_text(strip=True)

    snippet_element = result.select_one(".result__snippet")
    snippet = snippet_element.get_text(strip=True)

    # populate the new SERP result object and append it to the list. . .
    serp_result={
        "title": title,
        "url": url,
        "display_url": display_url,
        "snippet": snippet
    }
    serp_results.append(serp_result)

# print the extracted search results
for idx, result in enumerate(serp_results):
    print(f"Result {idx + 1}:")
    print(f"Title: {result['title']}")
    print(f"URL: {result['url']}")
    print(f"Display URL: {result['display_url']}")
    print(f"Snippet: {result['snippet']}\n")




Result 1:
Title: Stock Markets, Business News, Financials, Earnings - CNBC
URL: https://duckduckgo.com/l/?uddg=https%3A%2F%2Fwww.cnbc.com%2F&rut=e3cd8944b73e73f23d7d7c04ff125bfa1de1c58a9c6d6ff61a2ce7e480230ce1
Display URL: www.cnbc.com
Snippet: CNBC is the world leader in businessnewsand real-timefinancialmarket coverage. Find fast, actionable information.

Result 2:
Title: Yahoo Finance - Stock Market Live, Quotes, Business & Finance News
URL: https://duckduckgo.com/l/?uddg=https%3A%2F%2Ffinance.yahoo.com%2F&rut=24fcb7e68cbe4be4663c9e004873c06eaade06110b9e13494529e1d9fd48c349
Display URL: finance.yahoo.com
Snippet: At Yahoo Finance, you get free stock quotes, up-to-datenews, portfolio management resources, international market data, social interaction and mortgage rates that help you manage yourfinanciallife.

Result 3:
Title: Latest Finance News | Today's Top Headlines | Reuters
URL: https://duckduckgo.com/l/?uddg=https%3A%2F%2Fwww.reuters.com%2Fbusiness%2Ffinance%2F&rut=785ff1c50746

### Convert the links of financial news aggregator and title to csv

In [9]:
import csv

In [18]:
with open("financial_news_links.csv","w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["title", "url", "display_url", "snippet"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # write the header. . . 
    writer.writeheader()

    # write all the data rows. . .
    for result in serp_results:
        writer.writerow(result)

### From the news website fetch the news site present inside. . .

In [51]:
''' CNBC Scraping ONLY'''

' CNBC Scraping ONLY'

In [79]:
from scrapling.fetchers import FetcherSession
from urllib.parse import urljoin

base_url = "https://www.cnbc.com/"

with FetcherSession(impersonate="chrome") as session:
    page = session.get(base_url, stealthy_headers=True)
    hrefs = page.css("a::attr(href)").getall()

urls_cnbc = [urljoin(base_url, h) for h in hrefs if h]
# print(urls)

[2026-04-24 12:32:50] INFO: Fetched (200) <GET https://www.cnbc.com/world/?region=world> (referer: https://www.google.com/)


In [86]:
article_urls_cnbc = set()
for u in urls_cnbc:
    if (any(keyword in u for keyword in ["trading","stocks","finance"])) and "/2026" in u: # fetching the news articles of 2026:
        article_urls_cnbc.add(u)

for i in article_urls_cnbc:
    print(i)
# still a set. . . 

https://www.cnbc.com/2026/04/23/risky-meme-trading-is-back-a-trading-rule-change-may-have-lit-the-fuse.html
https://www.cnbc.com/2026/04/23/cramer-says-look-to-these-4-stocks-to-go-with-your-high-flying-tech-names.html
https://www.cnbc.com/2026/04/23/fridays-big-stock-stories-whats-likely-to-move-the-market-in-the-next-trading-session.html
https://www.cnbc.com/2026/04/24/european-stocks-iran-ceasefire-optimism-fades-.html
https://www.cnbc.com/2026/04/23/these-stocks-reporting-next-week-can-beat-expectations-and-rally-bespoke-says.html
https://www.cnbc.com/2026/04/23/stocks-making-the-biggest-moves-after-hours-intc-sap-byd-mxl.html
https://www.cnbc.com/2026/04/22/investment-strategies-ai-hardware-emerging-markets-uk-stocks.html


In [87]:
article_0=[]
for i in article_urls_cnbc:
    article_0.append(i)

'''Belongs to CNBC'''
for i in article_0:
    print(i)

# now a list for better handling. . .

https://www.cnbc.com/2026/04/23/risky-meme-trading-is-back-a-trading-rule-change-may-have-lit-the-fuse.html
https://www.cnbc.com/2026/04/23/cramer-says-look-to-these-4-stocks-to-go-with-your-high-flying-tech-names.html
https://www.cnbc.com/2026/04/23/fridays-big-stock-stories-whats-likely-to-move-the-market-in-the-next-trading-session.html
https://www.cnbc.com/2026/04/24/european-stocks-iran-ceasefire-optimism-fades-.html
https://www.cnbc.com/2026/04/23/these-stocks-reporting-next-week-can-beat-expectations-and-rally-bespoke-says.html
https://www.cnbc.com/2026/04/23/stocks-making-the-biggest-moves-after-hours-intc-sap-byd-mxl.html
https://www.cnbc.com/2026/04/22/investment-strategies-ai-hardware-emerging-markets-uk-stocks.html


In [88]:
''' AP News SCRAPING ONLY'''

' AP News SCRAPING ONLY'

In [62]:
from scrapling.fetchers import FetcherSession
from urllib.parse import urljoin

base_url = "apnews.com/hub/finance"

with FetcherSession(impersonate="chrome") as session:
    page = session.get(base_url, stealthy_headers=True)
    hrefs = page.css("a::attr(href)").getall()

urls_ft = [urljoin(base_url, h) for h in hrefs if h]

[2026-04-24 12:23:23] INFO: Fetched (200) <GET https://apnews.com/hub/finance> (referer: https://www.google.com/)


In [ ]:
urls_ft

['/',
 'https://apnews.com/world-news',
 'https://apnews.com/hub/iran',
 'https://apnews.com/hub/russia-ukraine',
 'https://apnews.com/hub/noticias',
 'https://apnews.com/hub/china',
 'https://apnews.com/hub/asia-pacific',
 'https://apnews.com/hub/latin-america',
 'https://apnews.com/hub/europe',
 'https://apnews.com/hub/africa',
 'https://apnews.com/article/us-iran-war-hormuz-israel-pakistan-ceasefire-april-23-2026-368b922ae2f4c874df8a133491eeffe8',
 'https://apnews.com/article/germany-iran-crown-prince-reza-pahlavi-liquid-0c2412ac58bb8e1b538c5e4f12abe381',
 'https://apnews.com/article/lebanon-israel-hezbollah-us-talks-ceasefire-washington-e7f26e207fc7543fe1f25a5318ff9ce3',
 '/newsletters?id=Morning+Wire+Subscribers',
 '/newsletters?id=Afternoon+Wire',
 '/newsletters',
 'https://apnews.com/us-news',
 'https://apnews.com/hub/immigration',
 'https://apnews.com/hub/weather',
 'https://apnews.com/education',
 'https://apnews.com/us-news/transportation',
 'https://apnews.com/hub/abortion',

In [ ]:
article_urls_ft=set()
for u in urls_ft:
    if (any(keyword in u for keyword in ["trading","stocks","finance","money","investment","economy","china-soft","japan","usa","china-tesla"])): # fetching the news articles of 2026
        article_urls_ft.add(u) # appending in sets to avoid duplicates

# for i in article_urls_ft:
#     print(i)

article_1=[]
for u in article_urls_ft:
    article_1.append(u) # converting set to list for better handling

for i in article_1:
    print(i)
    
'''Belongs to AP News'''

https://apnews.com/article/japan-trade-economy-oil-tariffs-trump-0dce5492a048ce92becbfbcf682067b9
https://apnews.com/article/china-soft-power-rise-c6aede1c6eb66a776a7ae3b5477e2661
https://apnews.com/article/shanghai-china-tesla-robots-electric-cars-musk-a05b41ae0d32fa391eaae1512871670a


In [82]:
'''Final for CNBC Finance'''

'Final for CNBC Finance'

In [83]:
from scrapling.fetchers import FetcherSession
from urllib.parse import urljoin

base_url = "www.cnbc.com/finance/"

with FetcherSession(impersonate="chrome") as session:
    page = session.get(base_url, stealthy_headers=True)
    hrefs = page.css("a::attr(href)").getall()

urls_cnbc_f = [urljoin(base_url, h) for h in hrefs if h]

[2026-04-24 12:36:46] INFO: Fetched (200) <GET https://www.cnbc.com/finance/> (referer: https://www.google.com/)


In [89]:
article_urls_cnbc_f = set()

for u in urls_cnbc_f:
    if (any(keyword in u for keyword in ["trading","stocks","finance"])) and "/2026" in u: # fetching the news articles of 2026:
        article_urls_cnbc_f.add(u)

article_2=[]
for i in article_urls_cnbc_f:
    article_2.append(i)

for i in article_2:
    print(i)

https://www.cnbc.com/2026/04/20/stocks-making-the-biggest-moves-premarket-aal-mrvl-mstr.html
https://www.cnbc.com/2026/04/20/stocks-making-the-biggest-moves-midday-swk-aal-dow-mrvl.html
https://www.cnbc.com/2026/04/23/risky-meme-trading-is-back-a-trading-rule-change-may-have-lit-the-fuse.html
https://www.cnbc.com/2026/04/20/stocks-making-the-biggest-moves-after-hours-amzn-aapl-stld.html
https://www.cnbc.com/2026/04/16/stocks-making-the-biggest-moves-premarket-.html
https://www.cnbc.com/2026/04/22/stocks-making-the-biggest-moves-after-hours-tsla-ibm-now-luv.html
https://www.cnbc.com/2026/04/21/stocks-making-the-biggest-moves-after-hours-adbe-ual-cof.html
https://www.cnbc.com/2026/04/19/software-tech-stocks-market-rally-microsoft.html
https://www.cnbc.com/2026/04/17/stocks-making-the-biggest-moves-premarket-nflx-orcl-afrm-and-more.html
https://www.cnbc.com/2026/04/20/howard-marks-says-there-are-very-few-cheap-stocks-bargains-come-when-people-panic.html
https://www.cnbc.com/2026/04/16/bur

### Fetch the content inside the child website. . 

In [103]:
articles=[]
temp=[article_0, article_1, article_2]
for i in temp:
    for j in i:
        articles.append(j)
        print(j)
    print("\n ==============")


https://www.cnbc.com/2026/04/23/risky-meme-trading-is-back-a-trading-rule-change-may-have-lit-the-fuse.html
https://www.cnbc.com/2026/04/23/cramer-says-look-to-these-4-stocks-to-go-with-your-high-flying-tech-names.html
https://www.cnbc.com/2026/04/23/fridays-big-stock-stories-whats-likely-to-move-the-market-in-the-next-trading-session.html
https://www.cnbc.com/2026/04/24/european-stocks-iran-ceasefire-optimism-fades-.html
https://www.cnbc.com/2026/04/23/these-stocks-reporting-next-week-can-beat-expectations-and-rally-bespoke-says.html
https://www.cnbc.com/2026/04/23/stocks-making-the-biggest-moves-after-hours-intc-sap-byd-mxl.html
https://www.cnbc.com/2026/04/22/investment-strategies-ai-hardware-emerging-markets-uk-stocks.html

https://apnews.com/article/japan-trade-economy-oil-tariffs-trump-0dce5492a048ce92becbfbcf682067b9
https://apnews.com/article/china-soft-power-rise-c6aede1c6eb66a776a7ae3b5477e2661
https://apnews.com/article/shanghai-china-tesla-robots-electric-cars-musk-a05b41ae

In [108]:
articles

['https://www.cnbc.com/2026/04/23/risky-meme-trading-is-back-a-trading-rule-change-may-have-lit-the-fuse.html',
 'https://www.cnbc.com/2026/04/23/cramer-says-look-to-these-4-stocks-to-go-with-your-high-flying-tech-names.html',
 'https://www.cnbc.com/2026/04/23/fridays-big-stock-stories-whats-likely-to-move-the-market-in-the-next-trading-session.html',
 'https://www.cnbc.com/2026/04/24/european-stocks-iran-ceasefire-optimism-fades-.html',
 'https://www.cnbc.com/2026/04/23/these-stocks-reporting-next-week-can-beat-expectations-and-rally-bespoke-says.html',
 'https://www.cnbc.com/2026/04/23/stocks-making-the-biggest-moves-after-hours-intc-sap-byd-mxl.html',
 'https://www.cnbc.com/2026/04/22/investment-strategies-ai-hardware-emerging-markets-uk-stocks.html',
 'https://apnews.com/article/japan-trade-economy-oil-tariffs-trump-0dce5492a048ce92becbfbcf682067b9',
 'https://apnews.com/article/china-soft-power-rise-c6aede1c6eb66a776a7ae3b5477e2661',
 'https://apnews.com/article/shanghai-china-tes

In [112]:
import trafilatura

url=articles[8] # belongs to CNBC
html = trafilatura.fetch_url(url)
text = trafilatura.extract(html)
print(url,end="\n\n")
print(text)

https://apnews.com/article/china-soft-power-rise-c6aede1c6eb66a776a7ae3b5477e2661

The ‘becoming Chinese’ meme shows China’s soft power moment is here
The ‘becoming Chinese’ meme shows China’s soft power moment is here
BANGKOK (AP) — Have you “become Chinese”?
In recent months, 20-somethings around the world have taken over social media with posts enthusing about how they’re embracing Chinese ways of life. Videos proclaiming users are “Chinamaxxing,” or “in a very Chinese time of their lives” — namely by drinking hot water with boiled goji berries, eating dumplings or wearing slippers in the house, or flying to China and gushing about its modern infrastructure — are racking up millions of views.
Along with its economic and geopolitical rise, China’s government has tried for years to push its soft power on the global stage. But those official efforts never came close to the success the “becoming Chinese” meme is enjoying now.
Even senior Chinese diplomats have noted the trend. Xie Feng,

In [ ]:
''' fetching all the article texts. . . 30 urls in total, we have. . .'''
article_text=[]

for i in articles:
    html = trafilatura.fetch_url(i)
    text = trafilatura.extract(html)
    article_text.append(text)


In [115]:
# create a csv containing the urls and their corresponding article texts. . .
import csv
with open("financial_news_articles.csv","w", newline="", encoding="utf-8-sig") as csvfile:
    fieldnames = ["url", "article_text"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # write the header. . . 
    writer.writeheader()

    # write all the data rows. . .
    for url, text in zip(articles, article_text):
        writer.writerow({"url": url, "article_text": text})

In [119]:
import pandas as pd

df=pd.read_csv("financial_news_articles.csv")

# count all the text in Article text column,
df["article_text"].str.len().sum()

np.int64(96056)

In [121]:
df.sample(5)

,url,article_text
26,https://www.cnbc.com/2026/04/21/stocks-making-...,Check out the companies making the biggest mov...
25,https://www.cnbc.com/2026/04/20/greg-abel-is-s...,Greg Abel is wasting little time putting his s...
8,https://apnews.com/article/china-soft-power-ri...,The ‘becoming Chinese’ meme shows China’s soft...
22,https://www.cnbc.com/2026/04/21/stocks-making-...,Check out the companies making headlines befor...
21,https://www.cnbc.com/2026/04/17/stocks-making-...,Check out the companies making the biggest mov...
